In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_books(url):
    # 1. Send an HTTP request to the website
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to retrieve the page. Status code: {response.status_code}")
        return []

    # 2. Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')
    books_data = []

    # 3. Extract product information
    # In this site, each book is inside an <article> tag with class 'product_pod'
    products = soup.find_all('article', class_='product_pod')

    for product in products:
        # Get Title
        title = product.h3.a['title']
        
        # Get Price
        price = product.find('p', class_='price_color').text
        
        # Get Rating (The class name is like 'star-rating Three')
        # We find the specific class that represents the rating
        rating_classes = product.find('p', class_='star-rating')['class']
        rating = rating_classes[1]  # Extracts 'Three', 'Four', etc.
        
        books_data.append({
            "Name": title,
            "Price": price,
            "Rating": rating
        })

    return books_data

# URL of the practice site
base_url = "http://books.toscrape.com/catalogue/page-1.html"
scraped_data = scrape_books(base_url)

# 4. Store the data in a CSV file
if scraped_data:
    df = pd.DataFrame(scraped_data)
    df.to_csv('products.csv', index=False, encoding='utf-8')
    print(f"Successfully scraped {len(scraped_data)} items and saved to 'products.csv'.")

Successfully scraped 20 items and saved to 'products.csv'.
